# 03 Calibration and conformal abstention

A router that says "0.92" should be right about 92% of the time when it says that, and it should know when to hand a message on.
This notebook measures how far the notebook-01 model is from that, fixes the scale with temperature scaling, and then builds the abstention gate:
a **conformal prediction set** decides the action. One route in the set: act. Several routes: escalate to the LLM. No route: a person looks.

What the guarantee covers: if calibration and live messages are exchangeable, the true route is in the set with probability at least 1 - alpha,
averaged over messages (marginal coverage). It does not promise that coverage holds for each route, and it degrades under drift. Both are shown below.

In [1]:
import json
import numpy as np
from pathlib import Path
from scipy.optimize import minimize_scalar
from scipy.special import softmax, logsumexp
from sklearn.linear_model import LogisticRegression
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import meridian as m

RESULTS = Path("data/results"); RESULTS.mkdir(parents=True, exist_ok=True)
FIG = Path("figures"); FIG.mkdir(exist_ok=True)
r01 = json.loads((RESULTS / "01_encoders_and_baseline.json").read_text())
ENC = r01["_best"]
print("encoder:", ENC, "| hardware:", m.hardware())

tr, ca, te = m.load("train"), m.load("calibration"), m.load("test")
_, ytr = m.texts_labels(tr); _, yca = m.texts_labels(ca); _, yte = m.texts_labels(te)
E = m.cached_embeddings(ENC)
clf = LogisticRegression(C=r01[ENC]["C"], max_iter=3000).fit(E["train"], ytr)
Z_ca, Z_te = clf.decision_function(E["calibration"]), clf.decision_function(E["test"])   # logits
P_ca, P_te = softmax(Z_ca, 1), softmax(Z_te, 1)
print(f"test accuracy {np.mean(P_te.argmax(1) == yte):.3f}")

encoder: bge-small | hardware: Intel(R) Core(TM) i7-7567U CPU @ 3.50GHz, 4 logical cores, Darwin 22.6.0


test accuracy 0.970


## Split the calibration data in two, by scenario
Temperature scaling fits one number; conformal calibration fits a quantile. Fitting both on the same 200 messages would let the first leak into the second,
so each gets its own half. Whole scenarios stay together, so the five variants of a situation never straddle the halves.

In [2]:
scen = np.array([f"{r['label']}:{r['scenario']}" for r in ca])
uniq = sorted(set(scen)); rng = np.random.default_rng(m.SEED); rng.shuffle(uniq)
half_T = np.isin(scen, uniq[: len(uniq) // 2]); half_C = ~half_T
print(f"temperature half: {half_T.sum()} messages; conformal half: {half_C.sum()} messages")

temperature half: 100 messages; conformal half: 100 messages


## Is the model calibrated? ECE depends on how you bin
Expected calibration error (ECE) groups predictions by confidence and compares each group's mean confidence with its accuracy.
The same model gets different ECE at 10, 15 and 30 bins, so the bin count is reported with the number.

**What the next cell does.** `ece` sorts the test predictions into confidence bins, compares each bin's mean confidence with its accuracy, and averages the gaps weighted by how many messages fall in the bin. `brier` is the mean squared distance between the probability vector and the true route (a one-hot vector). The cell prints ECE at 10, 15 and 30 bins.

In [3]:
def ece(P, y, bins):
    conf, pred = P.max(1), P.argmax(1); acc = (pred == y)
    edges = np.linspace(0, 1, bins + 1); tot = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        s = (conf > lo) & (conf <= hi)
        if s.any(): tot += s.mean() * abs(acc[s].mean() - conf[s].mean())
    return tot
def brier(P, y): return float(((P - np.eye(P.shape[1])[y]) ** 2).sum(1).mean())

out = {"encoder": ENC}
for b in (10, 15, 30): print(f"ECE, {b:2d} bins: {ece(P_te, yte, b):.3f}")
print(f"Brier score: {brier(P_te, yte):.3f}   mean confidence {P_te.max(1).mean():.3f} vs accuracy {np.mean(P_te.argmax(1) == yte):.3f}")
out["ece_before"] = {b: ece(P_te, yte, b) for b in (10, 15, 30)}; out["brier_before"] = brier(P_te, yte)

ECE, 10 bins: 0.440
ECE, 15 bins: 0.440
ECE, 30 bins: 0.440
Brier score: 0.299   mean confidence 0.530 vs accuracy 0.970


**Reading the output.** All three bin counts give 0.440, which is the gap between mean confidence (0.530) and accuracy (0.970): every prediction falls in one bin, so the bin count cannot matter. The head is underconfident. It says about 53% on average and is right 97% of the time, because `C=1` regularizes the weights toward zero and flattens the probabilities.

### The same measurement on a head fit with weak regularization
This head (`C=1000`) is overconfident instead. Because its confidences spread across the bins, the bin count now changes the number.

In [4]:
clf_hi = LogisticRegression(C=1000, max_iter=5000).fit(E["train"], ytr)
P_hi = softmax(clf_hi.decision_function(E["test"]), 1)
print(f"C=1000 head: accuracy {np.mean(P_hi.argmax(1) == yte):.3f}, mean confidence {P_hi.max(1).mean():.3f}")
for b in (10, 15, 30): print(f"   ECE, {b:2d} bins: {ece(P_hi, yte, b):.3f}")
out["ece_overconfident_head"] = {b: ece(P_hi, yte, b) for b in (10, 15, 30)}

C=1000 head: accuracy 0.970, mean confidence 0.955
   ECE, 10 bins: 0.017
   ECE, 15 bins: 0.027
   ECE, 30 bins: 0.037


**Reading the output.** The weakly regularized head is nearly calibrated on average (confidence 0.955, accuracy 0.970), and now ECE moves with the bin count, from 0.017 at 10 bins to 0.037 at 30. Report the bin count with every ECE.

## Temperature scaling
Divide the logits by one number T fitted on the temperature half by minimizing log loss. It cannot change which route wins, only how confident the model sounds.

**What the next cell does.** `nll` is the log loss of the logits after dividing by `T`. `minimize_scalar` finds the `T` with the lowest log loss on the temperature half only. The cell then applies that `T` to the test and calibration logits, prints ECE and Brier before and after, and saves a two-panel reliability plot to `figures/03_reliability.png` (points near the dashed diagonal mean confidence matches accuracy).

In [5]:
def nll(T, Z, y): return -(Z / T - logsumexp(Z / T, axis=1, keepdims=True))[np.arange(len(y)), y].mean()
T = minimize_scalar(lambda t: nll(t, Z_ca[half_T], yca[half_T]), bounds=(0.05, 20), method="bounded").x
Pt_te, Pt_ca = softmax(Z_te / T, 1), softmax(Z_ca / T, 1)
print(f"fitted T = {T:.2f} ({'T < 1 sharpens an underconfident model' if T < 1 else 'T > 1 softens an overconfident model'}; the head was fit with C={r01[ENC]['C']}, and a small C leaves the probabilities flat)")
for b in (10, 15, 30): print(f"ECE, {b:2d} bins: before {ece(P_te, yte, b):.3f} -> after {ece(Pt_te, yte, b):.3f}")
print(f"Brier: before {brier(P_te, yte):.3f} -> after {brier(Pt_te, yte):.3f}; accuracy unchanged: {np.mean(Pt_te.argmax(1) == yte):.3f}")
out["T"] = float(T); out["ece_after"] = {b: ece(Pt_te, yte, b) for b in (10, 15, 30)}; out["brier_after"] = brier(Pt_te, yte)

fig, ax = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
for a, (P, title) in zip(ax, [(P_te, "before"), (Pt_te, f"after temperature scaling (T={T:.2f})")]):
    conf, acc = P.max(1), (P.argmax(1) == yte); edges = np.linspace(0, 1, 11)
    xs, ys = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        s = (conf > lo) & (conf <= hi)
        if s.sum() >= 3: xs.append(conf[s].mean()); ys.append(acc[s].mean())
    a.plot([0, 1], [0, 1], "k--", lw=1); a.plot(xs, ys, "o-"); a.set_title(title); a.set_xlabel("confidence")
ax[0].set_ylabel("accuracy"); plt.tight_layout(); plt.savefig(FIG / "03_reliability.png", dpi=120); plt.close()

fitted T = 0.28 (T < 1 sharpens an underconfident model; the head was fit with C=1, and a small C leaves the probabilities flat)
ECE, 10 bins: before 0.440 -> after 0.044
ECE, 15 bins: before 0.440 -> after 0.044
ECE, 30 bins: before 0.440 -> after 0.046
Brier: before 0.299 -> after 0.059; accuracy unchanged: 0.970


**Reading the output.** `T` = 0.28 is below 1, so it sharpens the probabilities. ECE falls from 0.440 to 0.044 and Brier from 0.299 to 0.059, while accuracy stays at 0.970 because dividing the logits by a positive number never changes the largest one.

## Split conformal prediction
Nonconformity score: `1 - p(true route)`. On the conformal half, compute that score for every message; take the `ceil((n+1)(1-alpha))/n` quantile as the threshold `q`.
For a new message the prediction set is every route with `p >= 1 - q`.

**What the next cell does.** `conformal_q` sorts the nonconformity scores (`1 - p(true route)`) of the 100 conformal-half messages and returns the `k`-th smallest, with `k = ceil((n + 1)(1 - alpha))`. If `k` exceeds `n` it returns infinity, which puts every route in every set. `pred_sets` keeps each route whose probability is at least `1 - q`. `summarize` turns the sets into coverage with two intervals, the shares that act, escalate or go to a person, and the mean set size. `size_table` prints set sizes for several alphas on the temperature half, which the quantile never saw.

In [6]:
def conformal_q(P, y, alpha):
    s = np.sort(1 - P[np.arange(len(y)), y]); n = len(s)
    k = int(np.ceil((n + 1) * (1 - alpha)))
    return np.inf if k > n else s[k - 1]
def pred_sets(P, q): return P >= 1 - q          # boolean (n, routes)

n_c = half_C.sum(); print(f"conformal calibration set: {n_c} messages")
cl_te = m.cluster_ids(te)
def summarize(P, y, q, clusters, mask=None):
    S = pred_sets(P, q); size = S.sum(1); cov = S[np.arange(len(y)), y]
    if mask is not None: S, size, cov, y, clusters = S[mask], size[mask], cov[mask], y[mask], clusters[mask]
    n = len(y); k = int(cov.sum())
    lo, hi = m.wilson(k, n); clo, chi = m.cluster_bootstrap_ci(cov.astype(float), clusters)
    act = size == 1
    return dict(n=n, coverage=k / n, wilson=[lo, hi], cluster_ci=[clo, chi],
                frac_act=float(act.mean()), frac_escalate=float((size > 1).mean()), frac_human=float((size == 0).mean()),
                acc_when_act=float(cov[act].mean()) if act.any() else float("nan"), mean_size=float(size.mean()))

def size_table(P, y, alpha_list, Pcal, ycal):
    for a in alpha_list:
        q = conformal_q(Pcal, ycal, a); size = pred_sets(P, q).sum(1)
        print(f"alpha {a:.2f} (target coverage {1-a:.0%}): q={q:.3f} | sets of size 0: {(size==0).sum():3d}, size 1: {(size==1).sum():3d}, size 2 or more: {(size>=2).sum():3d}  (of {len(size)})")
print("set sizes on the temperature half (never seen by the conformal quantile):")
size_table(Pt_ca[half_T], yca[half_T], [0.01, 0.02, 0.05, 0.10], Pt_ca[half_C], yca[half_C])

conformal calibration set: 100 messages
set sizes on the temperature half (never seen by the conformal quantile):
alpha 0.01 (target coverage 99%): q=0.943 | sets of size 0:   0, size 1:  59, size 2 or more:  41  (of 100)
alpha 0.02 (target coverage 98%): q=0.849 | sets of size 0:   0, size 1:  71, size 2 or more:  29  (of 100)
alpha 0.05 (target coverage 95%): q=0.579 | sets of size 0:   5, size 1:  93, size 2 or more:   2  (of 100)
alpha 0.10 (target coverage 90%): q=0.202 | sets of size 0:  28, size 1:  72, size 2 or more:   0  (of 100)


**Reading the output.** The threshold `q` falls as alpha grows: 0.943 at alpha 0.01, 0.849 at 0.02, 0.202 at 0.10. A larger `q` admits more routes, so at alpha 0.01 41 of 100 messages get two or more routes and would go to the slow tier. At alpha 0.10, 28 get an empty set.

### Choosing alpha is a policy decision
Two things show in the table above. First, on this easy data a set with several routes appears only at a strict coverage target: at alpha 0.05 every set holds one route, and at 0.10 and above
a set holds one route or none, so only a strict target ever sends a message to the LLM. Second, the share of sets with two or more routes at the same alpha is very different on two 100-message
halves (20 scenarios each), so an operating point read off 100 calibration messages is unstable. Treat the escalation rate as an estimate with wide error.

We set **alpha = 0.02** (98% coverage) because a wrongly auto-approved refund costs several LLM calls, which is a cost argument, not a tuned value. Caveat: this notebook
had already printed the test curve below when alpha was fixed, so the cascade numbers in notebook 05 carry some optimism from that. A real project fixes alpha from the cost of an error
before the test split is opened.

In [7]:
ALPHA = 0.02
q = conformal_q(Pt_ca[half_C], yca[half_C], ALPHA)
r = summarize(Pt_te, yte, q, cl_te); out["alpha_chosen"] = ALPHA; out[f"conformal_alpha_{ALPHA}"] = {"q": float(q), **r}
print(f"test split at alpha={ALPHA}: target coverage {1-ALPHA:.2f}; q={q:.3f}")
print(f"   coverage {r['coverage']:.3f}  Wilson [{r['wilson'][0]:.3f}, {r['wilson'][1]:.3f}]  cluster [{r['cluster_ci'][0]:.3f}, {r['cluster_ci'][1]:.3f}]")
print(f"   act {r['frac_act']:.1%} (accuracy of those {r['acc_when_act']:.3f}) | escalate {r['frac_escalate']:.1%} | human {r['frac_human']:.1%} | mean set size {r['mean_size']:.2f}")

test split at alpha=0.02: target coverage 0.98; q=0.849
   coverage 0.985  Wilson [0.957, 0.995]  cluster [0.955, 1.000]
   act 91.5% (accuracy of those 0.984) | escalate 8.5% | human 0.0% | mean set size 1.09


**Reading the output.** On the test split the sets cover the true route for 98.5% of messages against a 98% target; both intervals (Wilson 0.957 to 0.995, cluster 0.955 to 1.000) contain 0.98. The encoder acts on 91.5% of messages and is right on 98.4% of those, 8.5% go to the slow tier, none to a person, and the mean set holds 1.09 routes.

## The coverage-versus-automation curve
Lower alpha demands more coverage, which means bigger sets, which means fewer messages handled automatically. The curve is the price list for that choice.

In [8]:
alphas = [0.005, 0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30]
curve = []
for a in alphas:
    q = conformal_q(Pt_ca[half_C], yca[half_C], a); r = summarize(Pt_te, yte, q, cl_te)
    curve.append((a, r["coverage"], r["frac_act"], r["acc_when_act"]))
    print(f"alpha {a:.2f}: coverage {r['coverage']:.3f} | automated {r['frac_act']:.1%} | accuracy of automated {r['acc_when_act']:.3f}")
out["curve"] = curve
plt.figure(figsize=(5, 4)); plt.plot([c[2] for c in curve], [c[3] for c in curve], "o-")
for a, _, x, y_ in curve: plt.annotate(f"{a}", (x, y_), fontsize=8)
plt.xlabel("fraction handled automatically"); plt.ylabel("accuracy of the automated slice"); plt.tight_layout(); plt.savefig(FIG / "03_coverage_automation.png", dpi=120); plt.close()

alpha 0.01: coverage 1.000 | automated 0.0% | accuracy of automated nan


alpha 0.01: coverage 1.000 | automated 77.5% | accuracy of automated 1.000


alpha 0.02: coverage 0.985 | automated 91.5% | accuracy of automated 0.984


alpha 0.05: coverage 0.970 | automated 100.0% | accuracy of automated 0.970


alpha 0.10: coverage 0.900 | automated 91.0% | accuracy of automated 0.989


alpha 0.15: coverage 0.860 | automated 87.0% | accuracy of automated 0.989


alpha 0.20: coverage 0.805 | automated 80.5% | accuracy of automated 1.000


alpha 0.30: coverage 0.690 | automated 69.0% | accuracy of automated 1.000


**Reading the output.** The curve is the price list for alpha: a stricter target hands more messages to the slow tier. The first line is alpha 0.005, printed as 0.01 because the format shows two decimals. With 100 calibration messages `k = ceil(101 x 0.995) = 101` exceeds 100, so `q` is infinite, every set holds all eight routes, coverage is 1.000 and nothing is automated. At alpha 0.05 every set holds exactly one route (100% automated, accuracy 0.970). In the size table above, from alpha 0.10 on the sets that are not automated are empty, which would go to a person, not to the slow tier.

## Where the guarantee breaks
Coverage is an average over messages, so it can hold overall and fail for a subgroup. Two checks on the chosen gate:
**per-route coverage**, and **coverage on boundary cases only** (messages that mention a neighboring route, a covariate shift toward hard inputs).

In [9]:
q10 = conformal_q(Pt_ca[half_C], yca[half_C], ALPHA)
S = pred_sets(Pt_te, q10); cov = S[np.arange(len(yte)), yte]
print(f"per-route coverage (target {1-ALPHA:.2f}, 25 messages each):")
for i, r_ in enumerate(m.ROUTES): print(f"   {r_:22s} {cov[yte == i].mean():.2f}")
bmask = np.array([r["boundary"] for r in te])
rb = summarize(Pt_te, yte, q10, cl_te, mask=bmask); rn = summarize(Pt_te, yte, q10, cl_te, mask=~bmask)
print(f"non-boundary messages (n={rn['n']}): coverage {rn['coverage']:.3f}, automated {rn['frac_act']:.1%}")
print(f"boundary messages     (n={rb['n']}): coverage {rb['coverage']:.3f}  Wilson [{rb['wilson'][0]:.3f}, {rb['wilson'][1]:.3f}], automated {rb['frac_act']:.1%}")
out["per_route_coverage"] = {m.ROUTES[i]: float(cov[yte == i].mean()) for i in range(8)}
out["boundary_shift"] = {"boundary": rb, "non_boundary": rn}

per-route coverage (target 0.98, 25 messages each):
   baggage_claim          1.00
   cancellation_request   1.00
   out_of_scope           1.00
   payment_dispute        1.00
   rebooking_request      1.00
   refund_claim           0.88
   service_complaint      1.00
   visa_entry_question    1.00


non-boundary messages (n=160): coverage 1.000, automated 92.5%
boundary messages     (n=40): coverage 0.925  Wilson [0.801, 0.974], automated 87.5%


**Reading the output.** Marginal coverage held (0.985) but one route did not: `refund_claim` is covered for 22 of its 25 messages (0.88), while the other seven routes are at 1.00. Messages written to sit near another route (boundary cases, n=40) are covered at 0.925 (Wilson 0.801 to 0.974), against 1.000 for the other 160.

### Label shift
Reweight the test set so one route makes up half of the traffic, as if a fare error produced a wave of refund claims. Marginal coverage is the weighted average of per-route coverage.

In [10]:
for tgt in ("refund_claim", "visa_entry_question"):
    j = m.ROUTES.index(tgt); w = np.where(yte == j, 7.0, 1.0); w = w / w.sum()
    print(f"traffic is {w[yte == j].sum():.0%} {tgt}: weighted coverage {np.sum(w * cov):.3f} (nominal {1-ALPHA:.3f})")
out["label_shift"] = {t: float(np.sum((np.where(yte == m.ROUTES.index(t), 7.0, 1.0) / np.where(yte == m.ROUTES.index(t), 7.0, 1.0).sum()) * cov)) for t in ("refund_claim", "visa_entry_question")}
(RESULTS / "03_calibration_and_conformal.json").write_text(json.dumps(out, indent=2, default=float))

traffic is 50% refund_claim: weighted coverage 0.940 (nominal 0.980)
traffic is 50% visa_entry_question: weighted coverage 0.991 (nominal 0.980)


2478

**Reading the output.** Marginal coverage is the average of the per-route coverages weighted by traffic share. If half the traffic is refund claims, the weak 0.88 counts half and coverage falls to 0.940 against the 0.980 target. If half is visa questions (covered at 1.00) it rises to 0.991. The guarantee was computed for the calibration mix and moves when the mix changes.